# Обработка geopqrquete файов для проверки скорости выполнения вычислений на batch

# Инициализация

In [1]:
import os
import sys

from sedona.spark import SedonaContext

# Явно указываем путь к Python для воркеров
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Настройка Hadoop
os.environ['HADOOP_HOME'] = r'C:\Hadoop\hadoop-3.3.6'

# Пакеты для Spark 3.5.4 со Scala 2.12
additional_packages = [
    "org.apache.sedona:sedona-spark-3.5_2.12:1.8.0",
    "org.datasyslab:geotools-wrapper:1.8.0-33.1"
]

config = SedonaContext.builder() \
    .appName("SedonaApp") \
    .config("spark.jars.packages", ",".join(additional_packages)) \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator") \
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions,org.apache.sedona.viz.sql.SedonaVizExtensions") \
    .config("spark.jars", r"D:\Artem\Work\amtech_projects\postgresql-42.7.13.jar") \
    .master("local[*]") \
    .getOrCreate()

sedona = SedonaContext.create(config)


# Декоратор для замера скорости выполнения запроса

In [2]:
import time
from functools import wraps

def timer_sedona(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"Функция {func.__name__} выполнена за {end - start:.4f} секунд")
        return result
    return wrapper

# Функция для выполнения запросов

In [13]:
@timer_sedona
def execute_query(session, query, geoparquet_file_path):
    loaded_df = sedona.read.format("geoparquet").load(geoparquet_file_path)
    # new_df = loaded_df.selectExpr(query)
    loaded_df.createOrReplaceTempView("spatial_table")
    result_df = session.sql(query)
    result_df.show(10)
    # return result_df

# Пути к geoparquet файлам

In [14]:
features_1_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_1.geoparquet"
features_10_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_10.geoparquet"
features_100_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_100.geoparquet"
features_1k_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_1k.geoparquet"
features_10k_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_10k.geoparquet"
features_100k_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_100k.geoparquet"

# Обработка всех файлов сразу

In [17]:
def execute_query_all(sedona, sql_area):
    print("features_1_path: ")
    execute_query(sedona, sql_area, features_1_path)
    print("\nfeatures_10_path: ")
    execute_query(sedona, sql_area, features_10_path)
    print("\nfeatures_100_path: ")
    execute_query(sedona, sql_area, features_100_path)
    print("\nfeatures_1k_path: ")
    execute_query(sedona, sql_area, features_1k_path)
    print("\nfeatures_10k_path: ")
    execute_query(sedona, sql_area, features_10k_path)
    print("\nfeatures_100k_path: ")
    execute_query(sedona, sql_area, features_100k_path)
    

# Вычисление площади

In [18]:
# запрос
sql_area = """SELECT *, ST_Area(geometry) as area FROM spatial_table"""
execute_query_all(sedona, sql_area)
# execute_query(sedona, sql_area, features_1_path)
# execute_query(sedona, sql_area, features_10_path)
# execute_query(sedona, sql_area, features_100_path)
# execute_query(sedona, sql_area, features_1k_path)
# execute_query(sedona, sql_area, features_10k_path)
# execute_query(sedona, sql_area, features_100k_path)

features_1_path: 
+-------+--------+--------------------+-------------------+
|     id|layer_id|            geometry|               area|
+-------+--------+--------------------+-------------------+
|3724141|     170|MULTIPOLYGON (((3...|6.79783493526394E-8|
+-------+--------+--------------------+-------------------+

Функция execute_query выполнена за 0.1212 секунд

features_10_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry|                area|
+-------+--------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...| 6.79783493526394E-8|
|3781881|     173|MULTIPOLYGON (((3...| 6.79783493526394E-8|
|3811204|     173|MULTIPOLYGON (((3...| 6.79783493526394E-8|
|3694818|     170|MULTIPOLYGON (((3...| 6.79783493526394E-8|
|3814588|     173|MULTIPOLYGON (((3...|2.771353487496294E-7|
|3692488|     170|MULTIPOLYGON (((3...|2.771353487496294E-7|
|3779551|     173|MULTIPOLYGON (((3...|2.771353487496294E-7|
|3